### Testing RAG Applications 📑

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "glm-4.7:cloud",
    temperature=0.5,
    max_tokens = 250
)

In [3]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


In [4]:
response = chain.invoke("What is MCP")

print(response)

Based on the provided context, the Model Context Protocol (MCP) is a protocol that standardizes LLM integration with external systems and utilizes a client-server architecture.


In [6]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [7]:
def query_with_context(question):
    retrieved_document = retrieve_and_format(question)
    response = qa_chain.run(question)
    return response, retrieved_document

In [8]:
actual, context = query_with_context("What is MCP")

actual, context

/var/folders/8h/zprf7hjs319_78816p34b90c0000gn/T/ipykernel_33611/1524904417.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain.run(question)


('Based on the provided context, MCP stands for the **Model Context Protocol**. It is a protocol that standardizes LLM (Large Language Model) integration with external systems using a client-server architecture. This enables quick and seamless communication between AI and external systems.',
 "IdentityDescope vs Okta CISDescope vs Amazon CognitoDescope vs StytchDescope vs WorkOSDescope vs FronteggApp Use CasesPasswordlessIdentity FederationATO PreventionIdentity OrchestrationAuthentication MethodsSocial LoginsPasskeysMFABiometrics / WebAuthnMagic LinksSSOOpenID ConnectnOTPOne-Time PasswordsAuthenticator AppsPasswordsDevelopersDocsTutorialsCommunityOpen SourceResourcesLearning CenterBlogCompanyOur StoryCareersPartnersNewsroomSecurity & ComplianceContact UsLegalPrivacy PolicyTerms of\n\n## Why this matters\n\nWhat Is the Model Context Protocol (MCP) and How It WorksSkip to main contentArrow RightGet your complimentary copy of the Gartner Report: IAM Adapts to Secure and Enable AI Agents.

### Testing RAG Application with RAGAs


In [9]:
test_data = [
    {
        "input": "What is MCP",
        "reference": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",
        "reference": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        "reference":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

In [10]:
dataset = []

for question in test_data:
    actual, context = query_with_context(question['input'])
    
    dataset.append({
        "user_input": question['input'],
        "retrieved_contexts": [context],
        "response": actual,
        "reference": question['reference']
    })

dataset


[{'user_input': 'What is MCP',
  'retrieved_contexts': ["IdentityDescope vs Okta CISDescope vs Amazon CognitoDescope vs StytchDescope vs WorkOSDescope vs FronteggApp Use CasesPasswordlessIdentity FederationATO PreventionIdentity OrchestrationAuthentication MethodsSocial LoginsPasskeysMFABiometrics / WebAuthnMagic LinksSSOOpenID ConnectnOTPOne-Time PasswordsAuthenticator AppsPasswordsDevelopersDocsTutorialsCommunityOpen SourceResourcesLearning CenterBlogCompanyOur StoryCareersPartnersNewsroomSecurity & ComplianceContact UsLegalPrivacy PolicyTerms of\n\n## Why this matters\n\nWhat Is the Model Context Protocol (MCP) and How It WorksSkip to main contentArrow RightGet your complimentary copy of the Gartner Report: IAM Adapts to Secure and Enable AI Agents. Let's go >Log InUser CircleProductUse CasesDevelopersCustomersResourcesCompanyPricingSign upArrow RightBook a demoArrow Right# Contents\n\n## Summary\nExplains the Model Context Protocol (MCP), how it standardizes LLM integration with ex

In [11]:
from ragas.metrics import LLMContextRecall, NoiseSensitivity, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas import (EvaluationDataset, evaluate)


evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(dataset)

result = evaluate(dataset=evaluation_dataset, 
                  metrics=[LLMContextRecall(),
                           Faithfulness(),
                           AnswerRelevancy(),
                           FactualCorrectness()],
                  llm = evaluator_llm)

/Users/vaibhavarde/Desktop/TestAutomation/TestAutomationSkills/llmEvaluation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/8h/zprf7hjs319_78816p34b90c0000gn/T/ipykernel_33611/3486702305.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, NoiseSensitivity, Faithfulness, FactualCorrectness, AnswerRelevancy
/var/folders/8h/zprf7hjs319_78816p34b90c0000gn/T/ipykernel_33611/3486702305.py:1: DeprecationWarning: Importing NoiseSensitivity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.met

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [18]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,factual_correctness(mode=f1)
0,What is MCP,"[reactions, retrieve channel history, and more...","MCP, or Model Context Protocol, is a protocol ...",The Model Context Protocol (MCP) addresses thi...,0.0,1.0,0.849991,0.17
1,What is Relationship between function calling ...,[then make the API call with it. While this al...,"Function calling, which allows Large Language ...",The Model Context Protocol (MCP) builds on top...,1.0,1.0,0.832278,1.00
2,"What are the core components of MCP, just give...",[understand that MCP doesn’t solve the NxM pro...,MCP Client & Server Ecosystem,\n - MCP Client\n ...,1.0,1.0,0.833868,0.50
